In [ ]:
import os
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from transformers import ViTImageProcessor, ViTForImageClassification
from sklearn.metrics import f1_score, confusion_matrix, classification_report
import numpy as np

# НАСТРОЙКИ:
model_path = "melanoma_attacks/checkpoint-12670"  # Путь к модели
original_dir = "balanced_Image"  # Путь к папке с оригинальными изображениями
attacked_dir = "isic_1000_attacked"  # Путь к папке с атакованными изображениями (ставить в зависимости от нужной атаки)
batch_size = 32

# ФУНКЦИИ:
def evaluate_dataset(data_dir, model, processor, device, batch_size=32):
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=processor.image_mean, std=processor.image_std)
    ])
    dataset = datasets.ImageFolder(data_dir, transform=transform)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images).logits
            preds = torch.argmax(outputs, dim=1).cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    f1 = f1_score(all_labels, all_preds, average="weighted")
    cm = confusion_matrix(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=dataset.classes)
    return  f1, cm, report, dataset.classes

# ЗАГРУЗКА МОДЕЛИ:
processor = ViTImageProcessor.from_pretrained(model_path)
model = ViTForImageClassification.from_pretrained(model_path)
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# ОЦЕНКА НА ОРИГИНАЛЬНЫХ:
print("=== ОЦЕНКА НА ОРИГИНАЛЬНЫХ ДАННЫХ ===")
orig_acc, orig_f1, orig_cm, orig_report, class_names = evaluate_dataset(original_dir, model, processor, device, batch_size)
print(f"F1-score: {orig_f1:.4f}")
print("Confusion Matrix:\n", orig_cm)
print("\nClassification Report:\n", orig_report)

# ОЦЕНКА НА АТАКОВАННЫХ:
print("\n=== ОЦЕНКА НА АТАКОВАННЫХ ДАННЫХ ===")
atk_acc, atk_f1, atk_cm, atk_report, _ = evaluate_dataset(attacked_dir, model, processor, device, batch_size)
print(f"F1-score: {atk_f1:.4f}")
print("Confusion Matrix:\n", atk_cm)
print("\nClassification Report:\n", atk_report)

# СРАВНЕНИЕ:
print("\n=== СРАВНЕНИЕ ===")
print(f"Падение F1-score: {orig_f1 - atk_f1:.4f}")
